## Zadania 8.1 i 8.2
 
**Numer grupy k:** 12

Ten raport opisuje część na 8 pkt: **aktualizację (8.1)** oraz **konflikt przy aktualizacji i front Pareto (8.2)**.
Zadania na 4 i 6 pkt są opisane w osobnym raporcie. Zadanie 8.3 (dane z PyPI) nie jest tu uwzględnione.

**Uruchamianie:** notebook uruchamiać z folderu `Project` (tam, gdzie leży `manager.py`).

## Spis treści
1. Idea rozwiązania
2. Zadanie 8.1 – aktualizacja
3. Zadanie 8.2 – konflikt i front Pareto
4. Testy i uruchamianie

## 1. Idea rozwiązania

Wszystkie operacje w projekcie mają tę samą budowę: **aksjomaty repozytorium (A1-A3) + twarde warunki + lista kryteriów minimalizowanych
leksykograficznie** (`z3.Optimize`, `priority="lex"`). Aktualizacja to nowa kombinacja twardych warunków i kryteriów, zapisana w
`use_cases/update.py`. Korzysta z gotowych elementów: `build_knowledge_base`, liczników z `objective_builders.py`, procedury `verify` i `explain`.

Przestarzałość wersji to jej pozycja na liście wersji pakietu od najnowszej (0 = najnowsza).

In [ ]:
import sys
from pathlib import Path
SRC = Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from package_manager.infrastructure.parsing import load_repository
from package_manager.infrastructure.state import parse_state
from package_manager.domain.verification import verify
from package_manager.use_cases.update import update, analyze_update

repo = load_repository("examples/repo_example.json")

def show(state):
    return {n: (str(v) + (" manual" if m else "")) for n, (v, m) in state.packages.items()}

## 2. Zadanie 8.1 – aktualizacja zbioru P

Funkcja `update(repo, state, package_names=None)`. Brak `package_names` oznacza wszystkie zainstalowane pakiety.
Stan początkowy jest najpierw weryfikowany (`verify`), a pakiet spoza stanu w P jest zgłaszany jako błąd.

**Twarde warunki:** aksjomaty repozytorium oraz to, że pakiety ręczne i pakiety z P pozostają zainstalowane.

**Kryteria (leksykograficznie, zgodnie z instrukcją):**

| # | Kryterium | Użyty licznik |
|---|---|---|
| 1 | minimalna przestarzałość pakietów z P | `obsolescence_cost` |
| 2 | minimalna liczba usuniętych pakietów | `count_unselected_packages` |
| 3 | minimalna liczba zmian wersji pakietów spoza P | `count_version_changes` |
| 4 | minimalna liczba nowych pakietów | `count_selected_packages` |
| 5 | minimalna przestarzałość wszystkich pakietów | `obsolescence_cost` |

Znaczniki `manual` są zachowane, nowe pakiety są automatyczne.

**CLI:** `python manager.py update [plik_z_nazwami_pakietów]`. Plik jest opcjonalny (nazwa pakietu w linii, `#` zaczyna komentarz);
po aktualizacji program wypisuje plan zmian i zapisuje stan.

### Wynik dla `state_example.txt` (aktualizacja wszystkiego)
```
Change plan:
- Upgrade lib 1.0 -> 1.5
- Install util 2.0
```
Stan końcowy: `app 1.0 manual`, `lib 1.5`, `util 2.0`, `tool 1.1 manual`, `json-slow 1.0`. To zgodne z tabelą w instrukcji.

**Dlaczego `lib 1.5`, a nie `2.0`?** `app 1.0` (ręczny, więc musi zostać) wymaga `lib <2.0`, a `tool 1.1` też. Najnowszy dozwolony `lib` to 1.5,
który wymaga dowolnego `util`. Kryterium 1 dotyczy tylko pakietów z P (tu wszystkich), więc `util` dostaje najnowszą wersję 2.0,
a że nie był zainstalowany, jest to jedyna nowa instalacja.

In [ ]:
state = parse_state(Path("examples/state_example.txt").read_text(encoding="utf-8"))
result = update(repo, state)
print(show(result))
print("verify:", verify(repo, result.get_packages_list()))

## 3. Zadanie 8.2 – konflikt przy aktualizacji i front Pareto

W `repo_example.json` najnowsze `a 2.0` wymaga `b <2.0`, a najnowsze `b 2.0` wymaga `a <2.0`, więc obu najnowszych nie da się mieć naraz.

Funkcja `analyze_update(repo, state, package_names=None)` działa w trzech krokach:

1. **Najnowsza osiągalna wersja każdego pakietu z P osobno** (`best_versions`). Dla każdego pakietu osobny `Optimize` z warunkiem
   „pakiety ręczne i ten pakiet zainstalowane” i minimalizacją jego przestarzałości.
2. **Test jednoczesności.** Żądania `p==najlepsza_wersja` (plus pakiety ręczne) trafiają do `explain`. Pusty rdzeń oznacza brak konfliktu.
3. **Przy konflikcie:**
   * przyczyna to minimalny rdzeń sprzeczności z `explain` (zadanie 6.3),
   * pakiety w konflikcie to te, których żądania wchodzą do rdzenia,
   * **front Pareto:** `Optimize` z `priority="pareto"` i osobnym `minimize` na przestarzałość każdego pakietu w konflikcie.
     Kolejne wywołania `check()` zwracają kolejne rozwiązania nieporównywalne (z limitem 50 rozwiązań).

Po wypisaniu konfliktu program stosuje kompromis wyznaczony przez kryteria z 8.1.

### Wynik: stan `a 1.0 manual`, `b 1.0 manual`, aktualizacja wszystkiego
```
Conflict: the newest reachable versions cannot be installed together.
Newest reachable separately: a 2.0, b 2.0
Reason (minimal unsatisfiable core):
- at most one package of b
- Package a==2.0 requires b <2.0
- Request #1: a==2.0
- Request #2: b==2.0
Pareto front (incomparable solutions):
- a 2.0, b 1.0
- a 1.1, b 2.0
Applying the best compromise (criteria of Task 8.1).
Change plan:
- Upgrade a 1.0 -> 1.1
- Upgrade b 1.0 -> 2.0
```
Front Pareto zgadza się z instrukcją: `{a 2.0, b 1.0}` i `{a 1.1, b 2.0}`. Rdzeń wyjaśnia przyczynę: `a 2.0` wymaga `b <2.0`,
a jednocześnie żądamy `b 2.0`, przy czym z pakietu `b` można mieć najwyżej jedną wersję.

### Wynik: aktualizacja samego `a`
```
Change plan:
- Upgrade a 1.0 -> 2.0
```
Bez konfliktu: `b` zostaje w wersji 1.0, bo `a 2.0` wymaga `b <2.0`, a `b` jest pakietem ręcznym spoza P.

### Uwaga o wyborze kompromisu
Oba rozwiązania z frontu mają tę samą sumę przestarzałości: `a 1.1` ma przestarzałość 1 i `b 2.0` ma 0, albo `a 2.0` ma 0 i `b 1.0` ma 1.
Kryteria z 8.1 nie rozstrzygają więc między nimi i wybór zależy od solvera. To zgodne z treścią zadania: suma przestarzałości jest
tylko jednym z możliwych kompromisów, dlatego program wypisuje cały front.

In [ ]:
ab = parse_state("a 1.0 manual\nb 1.0 manual\n")
report = analyze_update(repo, ab)
print("najnowsze osobno:", report["best"])
print("przyczyna:")
for label in report["core"]:
    print("  -", label)
print("front Pareto:", report["front"])

only_a = analyze_update(repo, ab, ["a"])
print("update samego a, brak konfliktu:", only_a["core"] == [])
print("wynik:", show(update(repo, ab, ["a"])))

## 4. Testy i uruchamianie

Testy w `tests/use_cases/test_update.py` sprawdzają:
* aktualizację wszystkiego dla `state_example.txt` (`lib 1.5`, `util 2.0`) i poprawność wyniku procedurą `verify`,
* zachowanie flag `manual`,
* aktualizację samego `a` (`a 2.0`, `b 1.0`, brak konfliktu),
* konflikt: najlepsze wersje `a 2.0, b 2.0`, niepusty rdzeń, front z dwoma rozwiązaniami,
* odrzucenie pakietu, który nie jest zainstalowany.

```bash
uv run python manager.py update            # aktualizacja wszystkiego
uv run python manager.py update p.txt      # aktualizacja pakietów z pliku
python -m pytest
```

In [ ]:
!python -m pytest tests/use_cases/test_update.py -q